# SAM3 — diagnose the edge constraint

One short **10-update native-loss adaptation**, using the original five training volumes.
Probe the same five adjacent-slice slabs at **updates 0, 5, and 10**. No baseline evaluation,
validation inference, constraint-training sweep, or new hyperparameter search is run.

The probe measures the native/bands/edge LoRA gradients, their alignment, the influence of
the existing coefficients, and gradients into mask, query-confidence and presence logits.
RNG and model buffers are restored after each probe so probing does not change adaptation.
The old calibration and completed results are read from their existing Drive folder.

**Run all** after refreshing this notebook. The original code/data/weights are reused.
All new files go to `diagnostics_edge_10updates_v1`. Completed diagnostics are skipped;
interrupt once and wait for a clean stop to resume safely. Do not run the old sweep concurrently.
Use the same GPU/software as the completed run (A100 80 GB in that run).

The previous notebook, including its outputs, is preserved separately. Existing validation
Dice: baseline 65.10%, bands 68.41%, bands+edge 52.00%.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED=time.monotonic()
HOME_DIR=Path('/content/drive/MyDrive/hippo_sam3_geofwi_exploration')
INPUT=HOME_DIR/'input'
SOURCE_RESULTS=HOME_DIR/'results_5vol_25updates_bands10_edge10_v1'
RESULTS=HOME_DIR/'diagnostics_edge_10updates_v1'
BUNDLE=Path('/content/hippo_sam3_geofwi_two_runs_bundle')
BASE_WEIGHTS=Path('/content/drive/MyDrive/hippo_medsam3_msd_extension/input/sam3.pt')
# Change BASE_WEIGHTS if your existing sam3.pt is elsewhere; it is never overwritten.
RESULTS.mkdir(parents=True,exist_ok=True)
print('Results and recovery:',RESULTS)


## Stage verified inputs on the local SSD
Files already present with matching hashes are reused. Different files cause a stop; active work is preserved.
The data ZIP contains only the exact five training and 96 validation volumes, not the full dataset.


In [ ]:
EXPECTED_UPLOADS={'sam3_geofwi_101vol_data.zip': '094456253ed3bf63e792d08b53eeb0c8bd36fc577a857c6cc6557d6f8271c5f0', 'sam3.pt': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e', 'sam3_geofwi_two_runs_code.zip': '3c7ea0c2f8b95d1dff0bd14d937f7d0238f8379b81ca07dc7fb1643a2b1f4536'}
EXPECTED_MANIFEST='7a8904c0151702fced8c1a922a760efe9f2c7b80afe0f599c1f2cd34b2612684'

import fcntl, shutil, zipfile
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def safe(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe ZIP path')
    return p
def install_zip(path,expected,allowed):
    if sha(path)!=expected:raise ValueError('ZIP hash mismatch: '+str(path))
    with zipfile.ZipFile(path) as archive:
        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError('Duplicate ZIP members')
        if set(archive.namelist())!=set(allowed):raise ValueError('ZIP inventory mismatch')
        for info in archive.infolist():
            safe(info.filename)
            if (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Symlink in ZIP')
        for name,expected_sha in allowed.items():
            target=BUNDLE/safe(name)
            if target.exists():
                if target.is_symlink() or sha(target)!=expected_sha:raise ValueError('Existing file differs: '+str(target))
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.staging')
            with archive.open(name) as source,tmp.open('wb') as dest:shutil.copyfileobj(source,dest)
            if sha(tmp)!=expected_sha:raise ValueError('Extracted member differs')
            tmp.replace(target)
(BUNDLE/'work').mkdir(parents=True,exist_ok=True)
with (BUNDLE/'work/sweep.lock').open('a') as lock:
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    package=INPUT/'sam3_geofwi_two_runs_code.zip'
    if sha(package)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Code ZIP changed')
    with zipfile.ZipFile(package) as archive:
        raw=archive.read('bundle_manifest.json')
        if hashlib.sha256(raw).hexdigest()!=EXPECTED_MANIFEST:raise ValueError('Scientific manifest differs')
        files=json.loads(raw);files['bundle_manifest.json']=EXPECTED_MANIFEST
    install_zip(package,EXPECTED_UPLOADS[package.name],files)
    data_files={'data/'+n:h for n,h in json.loads((BUNDLE/'data_manifest.json').read_text()).items()}
    if any(not (BUNDLE/n).exists() or sha(BUNDLE/n)!=h for n,h in data_files.items()):
        install_zip(INPUT/'sam3_geofwi_101vol_data.zip',EXPECTED_UPLOADS['sam3_geofwi_101vol_data.zip'],data_files)
    base=BUNDLE/'weights/sam3.pt';base.parent.mkdir(exist_ok=True)
    if not base.exists():
        if shutil.disk_usage(BUNDLE).free<16*1024**3:raise RuntimeError('Need16 GiB free runtime SSD before staging')
        if sha(BASE_WEIGHTS)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Existing sam3.pt differs')
        tmp=base.with_suffix('.staging');shutil.copyfile(BASE_WEIGHTS,tmp)
        if sha(tmp)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Checkpoint copy differs')
        tmp.replace(base)
    elif sha(base)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Local SAM3 checkpoint differs')
    provenance=RESULTS/'source'
    provenance.mkdir(exist_ok=True)
    dest=provenance/package.name
    if not dest.exists():shutil.copyfile(package,dest)
    if sha(dest)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Durable source package differs')
print('Verified frozen source,101 volumes and SAM3 weights on local SSD.')


## Environment and commands
Reuses matching installed versions; otherwise installs the same pinned dependencies as the MSD notebook.
An A100/H100-class BF16 GPU with at least 40 GB VRAM is required. Colab speed and CUDA equivalence
are checked on your actual GPU; local packaging tests cannot establish runtime speedups.


In [ ]:
VENV=Path('/content/hippo_sam3_geofwi_venv');PYTHON=VENV/'bin/python'
def setup(args):
    p=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (RESULTS/'setup.log').open('a') as log:log.write(p.stdout)
    print('\n'.join(p.stdout.splitlines()[-30:]))
    if p.returncode:raise RuntimeError('Setup failed; see setup.log')
if not PYTHON.exists():setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    path=Path('/content/sam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',path)
    setup([PYTHON,path])
probe="import importlib.metadata as m,sys; from pathlib import Path; lines=Path(sys.argv[1]).read_text().splitlines(); ok=all(m.version(s.split('==')[0])==s.split('==')[1] if '==' in s else bool(m.version(s)) for s in lines if s.strip()); sys.exit(0 if ok else 1)"
if subprocess.run([str(PYTHON),'-c',probe,str(BUNDLE/'requirements-colab.txt')],capture_output=True).returncode:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:print('Reusing matching dependency versions')
def execute(args,log_name,parallel=False):
    p=subprocess.Popen(list(map(str,args)),start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
                code=p.wait();break
            except KeyboardInterrupt:
                print('Clean stop requested. Keep this VM connected until the cell returns and recovery is verified.')
                if p.poll() is None:
                    if parallel:p.send_signal(signal.SIGUSR1)
                    else:os.killpg(p.pid,signal.SIGINT)
    if code:raise RuntimeError('Command failed; see '+str(RESULTS/log_name))
def campaign(command):
    execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS],command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Use the existing calibration

No full recalibration is run. The diagnostic verifies the completed archive, original initialization,
training support, saved coefficients, and GPU/software identity before adapting.


In [ ]:
DIAGNOSTIC_SOURCE=INPUT/'sam3_edge_diagnose_v1.py'
DIAGNOSTIC_SCRIPT=Path('/content/sam3_edge_diagnose_v1.py')
if sha(DIAGNOSTIC_SOURCE)!='2121b3831a71fbd5b17c966a9d701e73a1a2ee066dd1fb5921b1585014bde2ae':raise ValueError('Diagnostic source differs; refresh the notebook')
shutil.copyfile(DIAGNOSTIC_SOURCE,DIAGNOSTIC_SCRIPT)
if sha(DIAGNOSTIC_SCRIPT)!='2121b3831a71fbd5b17c966a9d701e73a1a2ee066dd1fb5921b1585014bde2ae':raise ValueError('Diagnostic copy failed')
print('Ready: 10 native updates; 5 support slabs at updates 0, 5, 10; no validation inference.')


## Run or resume the diagnostic

One worker runs at a time. The five fixed probes cover one slab from each training volume,
using anchors from the original calibration. Each probe reuses its forward graph for three
loss-gradient measurements. Checkpoints stay local each update and are copied to Drive at
updates 0, 5 and 10, or on a clean stop. Measurements are saved after every slab.

Interrupt once and wait for the cell to finish saving. A hard VM loss that loses charged updates
raises an error rather than silently repeating them. On completion, the verified small diagnostic
archive is retained and diagnostic recovery checkpoints are removed.


In [ ]:
execute([PYTHON,DIAGNOSTIC_SCRIPT,'--bundle',BUNDLE,
         '--source-results',SOURCE_RESULTS,'--durable',RESULTS],
        'diagnostic.log')


## Interpret the diagnosis

- **Edge/native+bands ratio:** the saved edge coefficient applied to the current gradients.
  The main table shows hypothetical full-strength ratios, so different probe stages are comparable.
  Per-slab JSON also records the original next-update warmup factor.
- **Conflicting slabs:** edge has a negative gradient dot product with native+bands.
- **Confidence derivatives:** a positive presence/query logit gradient pushes that confidence
  downward under direct gradient descent. These output-head derivatives are not additive
  percentages of the LoRA update; Adam and shared parameters can change the realized response.
- **Candidate weight:** what the original median/p95 rule suggests on these five probes with
  bands fixed. It is a diagnostic comparison, not a replacement for 20-slab production calibration.

This observes a **native-only adaptation trajectory**, not a replay of the failing edge trajectory.
Ratios growing during adaptation would support the calibration-drift hypothesis. Stable ratios
would weaken it. Five training slabs cannot establish validation benefit or prove causation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if not (RESULTS/'summary.json').exists():
    print('No probes saved yet; run or resume the diagnostic cell.')
else:
    summary=pd.DataFrame(json.loads((RESULTS/'summary.json').read_text()))
    display(summary)
    summary.to_csv(RESULTS/'summary.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for name,label in [('edge_ratio_median','Median'),('edge_ratio_p95','95th percentile'),('edge_ratio_max','Maximum')]:
        axes[0].plot(summary['update'],summary[name],marker='o',label=label)
    axes[0].axhline(.1,color='gray',linestyle='--',label='10% target')
    axes[0].axhline(.5,color='gray',linestyle=':',label='50% calibration p95 cap')
    axes[0].set(xlabel='Native adaptation updates',ylabel='Weighted edge / native+bands gradient norm')
    axes[0].legend()
    for name,label in [('mean_foreground_probability','Foreground probability'),('mean_presence_probability','Presence confidence')]:
        axes[1].plot(summary['update'],summary[name],marker='o',label=label)
    axes[1].set(xlabel='Native adaptation updates',ylabel='Probability',ylim=(0,1))
    axes[1].legend();fig.tight_layout();fig.savefig(RESULTS/'diagnostic.png',dpi=140);plt.show()
    status='complete' if (RESULTS/'completion.json').exists() else 'partial — resume the diagnostic cell'
    print('Diagnosis:',status)
    print('Detailed per-slab gradients and head derivatives:',RESULTS/'measurements.json')
